Kelly Quinn 
September 2026
"Does Outsourcing Still Cost Workers? Revisiting the Contractor Wage Penalty for Janitors and Security Guards, 2001–2025"

In [1]:
import numpy as np
import pandas as pd
from ipumspy import readers

ddi = readers.read_ipums_ddi("../data/cps_00047.xml")  # codebook only, fast
dk_raw  = pd.read_parquet("../data/dk_core.parquet")
dk_core = dk_raw[(dk_raw["AGE"] >= 16) & (dk_raw["YEAR"] <= 2025) & (dk_raw["YEAR"] != 2020)].copy()
print(dk_raw.shape, dk_core.shape)

/Users/kq/opt/anaconda3/envs/outsourcing/lib/python3.12/site-packages/ipumspy/readers.py:70: CitationWarning: Use of data from IPUMS is subject to conditions including that users should cite the data appropriately.
See the `ipums_conditions` attribute of this codebook for terms of use.
See the `ipums_citation` attribute of this codebook for the appropriate citation.
  warnings.warn(


(171905, 21) (166391, 21)


In [2]:
# check cells and compare to r df 
print(dk_core["YEAR"].min(), dk_core["YEAR"].max())
print(dk_core["OCC1990"].value_counts())
print((dk_core["EARNWT"] > 0).sum())
print(dk_core.loc[dk_core["EARNWT"] > 0, "MISH"].value_counts().sort_index())

1983 2025
OCC1990
453    122524
426     43867
Name: count, dtype: Int64
166391
MISH
4    82990
8    83401
Name: count, dtype: Int64


In [3]:
# codebook check before recoding 
for var in ["UHRSWORKORG", "UHRSWORK1", "HOURWAGE", "EARNWEEK2", "PAIDHOUR", "UNION", "METRO"]:
    print(var, ddi.get_variable_info(var).codes, "\n")

UHRSWORKORG {} 

UHRSWORK1 {'0 hours': 0, 'Hours vary': 997, 'NIU/Missing': 999} 

HOURWAGE {} 

EARNWEEK2 {} 

PAIDHOUR {'NIU': 0, 'No': 1, 'Yes': 2, 'Refused': 6, "Don't Know": 7} 

UNION {'NIU': 0, 'No union coverage': 1, 'Member of labor union': 2, 'Covered by union but not a member': 3} 

METRO {'Not identified': 0, 'Not in metropolitan area': 1, 'In central/principal city': 2, 'Not in central/principal city': 3, 'Central/principal city status not identified': 4, 'Missing/unknown': 9} 



In [4]:
for var in ["UHRSWORKORG", "HOURWAGE", "EARNWEEK2"]:
    s = dk_core[var]
    print(var, "max:", s.max())
    print(s.value_counts().sort_index().tail(5), "\n")

UHRSWORKORG max: 999
UHRSWORKORG
96         2
98         3
99        13
998        4
999    43254
Name: count, dtype: Int64 

HOURWAGE max: 999.99
HOURWAGE
98.00         1
98.85         1
99.00         2
99.50         1
999.99    29878
Name: count, dtype: int64 

EARNWEEK2 max: 999999.99
EARNWEEK2
9356.71      1
11256.27     1
11952.53     1
13258.23     2
999999.99    1
Name: count, dtype: int64 



## Build dk_clean: recode variables for eq. (1)

dk_core -> dk_clean. Constructs the outsourcing indicator, hourly wage, and all
controls used in the D&K baseline spec (union, part-time, age/age2, sex, race,
education categories, MSA/central city), plus the cluster variable (survey
month) and sample-period flags. No rows are dropped here — different
specifications use different subsets of covariates, so listwise deletion is
left to the estimation step.

In [5]:
# codebook check before recoding demographics/education
for var in ["RACE", "HISPAN", "EDUC", "SEX"]:
    print(var, ddi.get_variable_info(var).codes, "\n")

RACE {'White': 100, 'Black': 200, 'American Indian/Aleut/Eskimo': 300, 'Asian or Pacific Islander': 650, 'Asian only': 651, 'Hawaiian/Pacific Islander only': 652, 'Other (single) race, n.e.c.': 700, 'White-Black': 801, 'White-American Indian': 802, 'White-Asian': 803, 'White-Hawaiian/Pacific Islander': 804, 'Black-American Indian': 805, 'Black-Asian': 806, 'Black-Hawaiian/Pacific Islander': 807, 'American Indian-Asian': 808, 'Asian-Hawaiian/Pacific Islander': 809, 'White-Black-American Indian': 810, 'White-Black-Asian': 811, 'White-American Indian-Asian': 812, 'White-Asian-Hawaiian/Pacific Islander': 813, 'White-Black-American Indian-Asian': 814, 'American Indian-Hawaiian/Pacific Islander': 815, 'White-Black--Hawaiian/Pacific Islander': 816, 'White-American Indian-Hawaiian/Pacific Islander': 817, 'Black-American Indian-Asian': 818, 'White-American Indian-Asian-Hawaiian/Pacific Islander': 819, 'Two or three races, unspecified': 820, 'Four or five races, unspecified': 830, 'Blank': 999} 

In [6]:
dk_clean = dk_core.copy()

# missing-value sentinels -> NaN
dk_clean["HOURWAGE"] = dk_clean["HOURWAGE"].replace(999.99, np.nan)
dk_clean["EARNWEEK2"] = dk_clean["EARNWEEK2"].replace(999999.99, np.nan)
dk_clean["UHRSWORKORG"] = dk_clean["UHRSWORKORG"].replace([998, 999], np.nan)
dk_clean["UHRSWORK1"] = dk_clean["UHRSWORK1"].replace([997, 999], np.nan)

# UHRSWORKORG is the hours variable matched to ORG earnings; fall back to UHRSWORK1
# only when it's missing, to reduce dropped observations
dk_clean["hours"] = dk_clean["UHRSWORKORG"].fillna(dk_clean["UHRSWORK1"])

# hourly wage: reported rate for workers paid hourly, else weekly earnings / usual hours.
# 0 hours can't be used as a divisor, so it's treated as missing too; fillna(0) on the
# condition (rather than a bare > 0) avoids pd.NA propagating into np.where's condition array
dk_clean["hourly_wage"] = np.where(
    dk_clean["PAIDHOUR"] == 2,
    dk_clean["HOURWAGE"],
    np.where(dk_clean["hours"].fillna(0) > 0, dk_clean["EARNWEEK2"] / dk_clean["hours"], np.nan),
)
dk_clean["log_wage"] = np.log(dk_clean["hourly_wage"].where(dk_clean["hourly_wage"] > 0))

print(dk_clean[["hours", "hourly_wage", "log_wage"]].describe())

           hours    hourly_wage       log_wage
count   164041.0  165133.000000  159224.000000
mean   34.561701      10.106066       2.184729
std    11.241157       8.546088       0.561265
min          0.0       0.000000      -4.605170
25%         30.0       5.950000       1.791759
50%         40.0       8.700000       2.197225
75%         40.0      12.500000       2.525729
max         99.0     750.000000       6.620073


In [7]:
# outsourcing status: janitors (453) in Services to Buildings & Dwellings (722),
# guards (426) in Protective Services (740)
dk_clean["outsourced"] = (
    ((dk_clean["OCC1990"] == 453) & (dk_clean["IND1990"] == 722))
    | ((dk_clean["OCC1990"] == 426) & (dk_clean["IND1990"] == 740))
).astype(int)
dk_clean["occupation"] = np.where(dk_clean["OCC1990"] == 453, "janitor", "guard")

dk_clean["union"] = dk_clean["UNION"].map({1: 0, 2: 1, 3: 1})  # member or covered; NIU (0) -> NaN
dk_clean["parttime"] = np.where(
    dk_clean["hours"].isna(), np.nan, (dk_clean["hours"] < 30).astype(float)
)

print(dk_clean.groupby("occupation")["outsourced"].mean())
print(dk_clean[["union", "parttime"]].isna().sum())

occupation
guard      0.436456
janitor    0.184903
Name: outsourced, dtype: float64
union       1698
parttime    2350
dtype: int64


In [8]:
dk_clean["female"] = (dk_clean["SEX"] == 2).astype(int)
dk_clean["black"] = (dk_clean["RACE"] == 200).astype(int)  # single-race Black; matches D&K's 1983-2000-era coding
dk_clean["hispanic"] = (~dk_clean["HISPAN"].isin([0, 901, 902])).astype(int)
dk_clean["age"] = dk_clean["AGE"]
dk_clean["age2"] = dk_clean["age"] ** 2

# D&K's six education categories (college completion is the omitted/reference category)
educ_bins = {
    "no_school": [2],
    "primary": [10, 11, 12, 13, 14, 20, 21, 22, 30, 31, 32],
    "hs_attend": [40, 50, 60, 71, 72],
    "hs_complete": [73],
    "some_college": [80, 81, 90, 91, 92, 100],
    "college": [110, 111, 120, 121, 122, 123, 124, 125],
}
dk_clean["educ_cat"] = pd.NA
for label, codes in educ_bins.items():
    dk_clean.loc[dk_clean["EDUC"].isin(codes), "educ_cat"] = label
dk_clean["educ_cat"] = pd.Categorical(dk_clean["educ_cat"], categories=list(educ_bins), ordered=False)

print(dk_clean["educ_cat"].value_counts(dropna=False))

educ_cat
hs_complete     56335
hs_attend       45299
some_college    36290
primary         15994
college         11477
no_school         996
Name: count, dtype: int64


In [19]:
# MSA / central city status; METRO 4 = in MSA, central city status unknown (see CLAUDE.md)
# (.isin() used throughout instead of == : nullable Int64 columns return a masked
# BooleanArray from ==, which np.select rejects)
dk_clean["msa"] = np.select(
    [dk_clean["METRO"].isin([0, 9]), dk_clean["METRO"].isin([1]), dk_clean["METRO"].isin([2, 3, 4])],
    [np.nan, 0, 1],
    default=np.nan,
)
dk_clean["central_city"] = np.select(
    [dk_clean["METRO"].isin([0, 9, 4]), dk_clean["METRO"].isin([2]), dk_clean["METRO"].isin([1, 3])],
    [np.nan, 1, 0],
    default=np.nan,
)

dk_clean["ym"] = dk_clean["YEAR"] * 100 + dk_clean["MONTH"]  # cluster variable (survey month)
dk_clean["replication_sample"] = dk_clean["YEAR"].between(1983, 2000)
dk_clean["extension_sample"] = dk_clean["YEAR"] >= 2001  # 2020 already excluded in dk_core

reg_vars = [
    "log_wage", "outsourced", "union", "parttime", "age", "age2", "female",
    "black", "hispanic", "educ_cat", "msa", "central_city", "STATEFIP", "YEAR",
    "ym", "EARNWT",
]
print(dk_clean.shape)
print(dk_clean[reg_vars].isna().sum())

dk_clean.to_parquet("../data/dk_clean.parquet")

(166391, 39)
log_wage         7167
outsourced          0
union            1698
parttime         2350
age                 0
age2                0
female              0
black               0
hispanic            0
educ_cat            0
msa             14585
central_city    30663
STATEFIP            0
YEAR                0
ym                  0
EARNWT              0
dtype: int64


In [18]:
metro_cat = dk_clean["METRO"].map({
    1: "nonmetro",           # omitted category
    2: "metro_cc",
    3: "metro_not_cc",
    4: "metro_cc_unknown",
    0: "not_identified",
    9: "not_identified",
})

pd.crosstab(metro_cat, dk_clean["outsourced"], normalize="columns")

pd.crosstab([dk_clean["OCC1990"], dk_clean["YEAR"] >= 2004, dk_clean["outsourced"]],
            metro_cat, normalize="index").round(3)

dk_clean.groupby([metro_cat, "outsourced"])["log_wage"].mean().unstack()

wage_sample = dk_clean[dk_clean["log_wage"].notna()]
demeaned = (wage_sample["log_wage"]
            - wage_sample.groupby(["OCC1990", "YEAR"])["log_wage"].transform("mean"))

(demeaned
 .groupby([wage_sample["OCC1990"], metro_cat, wage_sample["outsourced"]])
 .mean()
 .unstack()
 .round(3))

outsourced                    0      1
OCC1990 METRO                         
426     metro_cc          0.084 -0.100
        metro_cc_unknown  0.059 -0.104
        metro_not_cc      0.138 -0.093
        nonmetro          0.008 -0.183
        not_identified    0.066 -0.193
453     metro_cc          0.024 -0.089
        metro_cc_unknown  0.014 -0.121
        metro_not_cc      0.071 -0.079
        nonmetro         -0.022 -0.130
        not_identified    0.003 -0.188